# Lektion 4: Webscraping mit trafilatura – Colab-Version

**Datenmanagement für die MT**

Dieses Notebook ist der **Plan B** für alle, bei denen Python auf dem eigenen Computer nicht läuft. Es macht genau dasselbe wie die Skripte `texte_herunterladen.py` und `regex_suche.py` – nur im Browser. Die Anleitung zur Übung finden Sie weiterhin im Wiki unter [4.4 Übung](https://github.com/mkappus1/DatenmanagementMT2026/wiki/4.4-%C3%9Cbung-Mein-zweisprachiges-Mini-Korpus). Dieses Notebook ersetzt nur die Arbeit im Terminal.

## So funktioniert Colab

- Ein Notebook besteht aus **Zellen**. Textzellen (wie diese) erklären, **Codezellen** (grau hinterlegt) führen etwas aus.
- Eine Codezelle starten Sie mit dem **▶-Symbol** links neben der Zelle oder mit **Shift + Enter**.
- Führen Sie die Zellen **von oben nach unten** aus. Jede Zelle baut auf den vorherigen auf.
- Beim ersten Start fragt Colab eventuell, ob Sie dem Notebook vertrauen. Klicken Sie auf **«Trotzdem ausführen»**.
- Ihre Dateien sehen Sie über das **Ordnersymbol 📁** in der linken Seitenleiste.

> ⚠️ **Wichtig:** Colab löscht alle Dateien, sobald Sie das Notebook schliessen oder längere Zeit nichts tun. Laden Sie Ihre Ergebnisse deshalb am Schluss herunter (siehe letzter Abschnitt). Wenn Colab die Verbindung verloren hat, führen Sie Schritt 1 einfach nochmals aus.

## Schritt 1: Vorbereitung

Diese beiden Zellen installieren trafilatura und laden die Werkzeuge. Führen Sie beide nacheinander aus. Die erste dauert etwa eine halbe Minute.

In [ ]:
!pip install -q trafilatura
print("trafilatura ist installiert.")

In [ ]:
#@title Werkzeuge laden (Code ausgeblendet – hier müssen Sie nichts ändern)
import re
import time
from pathlib import Path

import trafilatura
from lxml import html as lxml_html

# Bereiche, die nie zum Haupttext gehören: Such-Overlays, Seitenkopf und -fuss
# (ausser innerhalb eines Artikels), Navigation, Schaltflächen, Formulare.
BOILERPLATE = (
    "//*[@role='search'] | //form | //nav | //button"
    " | //header[not(ancestor::article) and not(ancestor::main)]"
    " | //footer[not(ancestor::article) and not(ancestor::main)]"
)


def _ist_webadresse(text):
    return text.strip().lower().startswith(("http://", "https://"))


def _html_vorbereiten(html):
    try:
        dokument = lxml_html.fromstring(html)
    except Exception:
        return html
    for element in dokument.xpath(BOILERPLATE):
        if element.getparent() is not None:
            element.drop_tree()
    for element in dokument.xpath("//*[@aria-hidden='true']"):
        del element.attrib["aria-hidden"]
    for element in dokument.xpath("//label[@aria-controls] | //summary"):
        element.tag = "h3"
    return lxml_html.tostring(dokument, encoding="unicode")


def _text_herunterladen(adresse):
    html = trafilatura.fetch_url(adresse)
    if html is None:
        print("      Fehler: Seite konnte nicht geladen werden.")
        print("      Bitte Adresse im Browser testen und neu hineinkopieren.")
        return None
    text = trafilatura.extract(_html_vorbereiten(html), output_format="txt",
                               include_comments=False, include_tables=True)
    if not text or not text.strip():
        print("      Fehler: Kein Text gefunden.")
        print("      trafilatura konnte auf dieser Seite keinen Haupttext erkennen.")
        return None
    zeilen = [z.rstrip() for z in text.splitlines() if z.strip()]
    metadaten = trafilatura.extract_metadata(html)
    titel = metadaten.title.strip() if metadaten and metadaten.title else ""
    if titel and zeilen[0] != titel:
        zeilen.insert(0, titel)
    return "\n".join(zeilen)


def herunterladen(QUELLE, AUSGABE, PAUSE=2):
    """Entspricht dem Skript texte_herunterladen.py."""
    if _ist_webadresse(QUELLE):
        adressen = [QUELLE.strip()]
    else:
        datei = Path(QUELLE)
        if not datei.is_file():
            print(f"Fehler: Die Datei '{QUELLE}' wurde nicht gefunden.")
            print("Haben Sie die Zelle, die die Liste anlegt, schon ausgeführt?")
            return
        zeilen = datei.read_text(encoding="utf-8-sig").splitlines()
        adressen = [z.strip() for z in zeilen if _ist_webadresse(z)]
        if not adressen:
            print(f"Fehler: In der Datei '{QUELLE}' steht keine Webadresse.")
            return
    aus_liste = not _ist_webadresse(QUELLE)
    anzahl = len(adressen)
    print(f"{anzahl} Seite(n) werden heruntergeladen ...\n")
    texte = []
    for nr, adresse in enumerate(adressen, start=1):
        if nr > 1:
            time.sleep(float(PAUSE))  # Fairness: Server nicht überlasten
        print(f"[{nr}/{anzahl}] {adresse}")
        text = _text_herunterladen(adresse)
        if text is None:
            continue
        print(f"      OK: {len(text.splitlines())} Zeilen Text")
        texte.append(f"### {adresse}\n{text}" if aus_liste else text)
    print()
    if not texte:
        print("Es konnte keine Seite gespeichert werden. Die Ausgabedatei wurde nicht angelegt.")
        return
    Path(AUSGABE).write_text("\n".join(texte) + "\n", encoding="utf-8")
    print(f"Fertig: {len(texte)} von {anzahl} Seite(n) gespeichert in {AUSGABE}")


def regex_suche(DATEI, MUSTER, MODUS, ERSATZ, AUSGABE):
    """Entspricht dem Skript regex_suche.py aus Lektion 3."""
    if MODUS not in ("zeilen", "treffer", "ersetzen"):
        print('Fehler: MODUS muss "zeilen", "treffer" oder "ersetzen" sein.')
        return
    if not Path(DATEI).is_file():
        print(f"Fehler: Die Datei '{DATEI}' wurde nicht gefunden.")
        return
    text = Path(DATEI).read_text(encoding="utf-8-sig")
    if MODUS == "zeilen":
        ergebnis = [z for z in text.splitlines() if re.search(MUSTER, z)]
    elif MODUS == "treffer":
        ergebnis = [m.group(0) for m in re.finditer(MUSTER, text)]
    else:
        neu, anzahl = re.subn(MUSTER, ERSATZ, text)
        ergebnis = [neu.rstrip("\n")]
    inhalt = "\n".join(ergebnis)
    Path(AUSGABE).write_text(inhalt + "\n" if inhalt else "", encoding="utf-8")
    if MODUS == "ersetzen":
        print(anzahl, "Ersetzungen gespeichert in", AUSGABE)
    else:
        print(len(ergebnis), "Treffer gespeichert in", AUSGABE)


print("Werkzeuge geladen. Sie können jetzt weitermachen.")

## Schritt 2: Eine Seite herunterladen

Diese Zelle entspricht dem Skript `texte_herunterladen.py`. Wie im Skript ändern Sie nur die drei Parameter. Die Anführungszeichen bleiben stehen.

| Parameter | Bedeutung |
|---|---|
| `QUELLE` | eine Webadresse (beginnt mit `https://`) oder der Name einer Textdatei mit mehreren Webadressen |
| `AUSGABE` | Name der Textdatei, in die der Text gespeichert wird |
| `PAUSE` | Sekunden Wartezeit zwischen zwei Seiten |

Für die **zweite Sprache** ändern Sie `QUELLE` und `AUSGABE` und führen die Zelle nochmals aus.

In [ ]:
# ---- Parameter anpassen ----
QUELLE  = "https://www.ebgb.admin.ch/de/inklusions-initiative-in-leichter-sprache-die-antwort-vom-bundesrat-in-leichter-sprache"
AUSGABE = "inklusion_de.txt"
PAUSE   = 2
# ----------------------------

herunterladen(QUELLE, AUSGABE, PAUSE)

## Schritt 3: Das Ergebnis anschauen

In Colab funktionieren dieselben Befehle wie im Mac-Terminal. Sie müssen nur ein Ausrufezeichen `!` davor setzen. Passen Sie den Dateinamen jeweils an.

**Inhalt anzeigen:**

In [ ]:
!cat inklusion_de.txt

**Zeilen zählen** (für mehrere Dateien die Namen mit Leerzeichen getrennt angeben):

In [ ]:
!wc -l inklusion_de.txt

> **Dateien bearbeiten:** Wenn Sie in einer Datei Zeilen löschen möchten (Übung 4.4, Teil A, Schritt 5a), öffnen Sie links das Ordnersymbol 📁 und klicken Sie doppelt auf die Datei. Sie öffnet sich in einem Editor rechts. Änderungen werden automatisch gespeichert.

## Schritt 4: Suchen und Ersetzen mit regulären Ausdrücken

Diese Zelle entspricht dem Skript `regex_suche.py` aus Lektion 3. Die Parameter sind dieselben:

| Parameter | Bedeutung |
|---|---|
| `DATEI` | die Datei, in der gesucht wird |
| `MUSTER` | der reguläre Ausdruck (das `r` vor den Anführungszeichen bleibt stehen) |
| `MODUS` | `"zeilen"`, `"treffer"` oder `"ersetzen"` |
| `ERSATZ` | der Ersatztext, nur für den Modus `"ersetzen"` |
| `AUSGABE` | die Datei, in die das Ergebnis geschrieben wird |

Voreingestellt ist das Entfernen der Aufzählungszeichen aus Übung 4.4.

In [ ]:
# ---- Parameter anpassen ----
DATEI   = "inklusion_de.txt"
MUSTER  = r"(?m)^- "
MODUS   = "ersetzen"      # "zeilen", "treffer" oder "ersetzen"
ERSATZ  = ""              # nur für MODUS "ersetzen"
AUSGABE = "inklusion_de_sauber.txt"
# ----------------------------

regex_suche(DATEI, MUSTER, MODUS, ERSATZ, AUSGABE)

## Schritt 5: Mehrere Seiten auf einmal (Teil B)

Im Terminal legen Sie die URL-Listen im Editor an. In Colab geht das mit der folgenden Zelle: Die erste Zeile `%%writefile urls_de.txt` bedeutet «schreibe alles darunter in die Datei `urls_de.txt`». Ersetzen Sie die Beispieladressen durch Ihre eigenen, eine pro Zeile, und führen Sie die Zelle aus.

Für die zweite Sprache ändern Sie den Dateinamen in der ersten Zeile (z. B. `urls_fr.txt`) und die Adressen.

⚠️ Die erste Zeile `%%writefile …` muss ganz oben in der Zelle stehen – ohne Leerzeile davor.

In [ ]:
%%writefile urls_de.txt
https://www.ebgb.admin.ch/de/inklusions-initiative-in-leichter-sprache-die-antwort-vom-bundesrat-in-leichter-sprache
https://www.ebgb.admin.ch/de/gesundheit-fuer-alle

Jetzt laden Sie alle Seiten der Liste herunter. Das dauert wegen der Pause zwischen den Seiten etwas länger:

In [ ]:
# ---- Parameter anpassen ----
QUELLE  = "urls_de.txt"
AUSGABE = "korpus_de.txt"
PAUSE   = 2
# ----------------------------

herunterladen(QUELLE, AUSGABE, PAUSE)

**Sind alle Texte da?** Zählen Sie die `###`-Zeilen (ein Text = eine `###`-Zeile):

In [ ]:
!grep -c "^###" korpus_de.txt

**Wie gross ist das Korpus?** Zeilen und Wörter zählen:

In [ ]:
!wc -l -w korpus_de.txt

## Schritt 6: Ergebnisse herunterladen

Tragen Sie in der Liste alle Dateien ein, die Sie abgeben möchten, und führen Sie die Zelle aus. Ihr Browser lädt die Dateien dann einzeln herunter. Falls er fragt, ob mehrere Downloads erlaubt sind, bestätigen Sie mit **«Zulassen»**.

In [ ]:
# ---- Parameter anpassen ----
DATEIEN = [
    "inklusion_de_sauber.txt",
    "inklusion_fr_sauber.txt",
    "korpus_de.txt",
    "korpus_fr.txt",
]
# ----------------------------

from google.colab import files
for name in DATEIEN:
    if Path(name).is_file():
        files.download(name)
    else:
        print(f"Datei nicht gefunden (wird übersprungen): {name}")